# NB 8 — An orchestrator with specialist sub-agents
**Goal:** show multi-agent coordination *and* its cost. An **orchestrator** decomposes a request, routes sub-tasks to specialist sub-agents, then **aggregates and verifies** — and we run the *same* task with one well-instrumented agent so the team can feel when the extra machinery is, and isn't, worth it.

The task: prepare the post-discharge anticoagulant follow-up. (Runs in MOCK mode with no API key.)

In [1]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model: pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages, temperature=0):
    "Route by the specialist role named in the system prompt."
    sys = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    if "triage"     in sys: return "urgency=HIGH; INR 4.2 is well above range with recent DVT."
    if "records"    in sys: return "INR 4.2 (range 2.0-3.0); meds: warfarin 5 mg daily, lisinopril 10 mg."
    if "scheduling" in sys: return "Earliest anticoagulation-clinic slot: in 2 days (Thu 10:00)."
    # single-agent path: produce the whole plan at once
    return ("Plan: flag out-of-range INR (4.2) for clinician dose review; bring the anticoagulation "
            "follow-up forward to ~2 days; no order changed by the agent.")

Backend: REAL model = openai/gpt-4o-mini


### Three specialist sub-agents
Each is just the model with a focused role prompt and a narrow job.

In [2]:
def specialist(role, instruction, context):
    sys = f"You are the {role} sub-agent. Do only your part. Be terse."
    return chat([{"role":"system","content":sys},
                 {"role":"user","content":f"{instruction}\nContext: {context}"}])

PATIENT = ("Discharged 5 days ago after DVT; on warfarin 5 mg daily and lisinopril 10 mg daily; "
           "INR resulted today at 4.2 (reference range 2.0-3.0).")
def triage_agent(c):    return specialist("triage",     "Assess urgency.", c)
def records_agent(c):   return specialist("records",    "Pull the INR and medications.", c)
def scheduling_agent(c):return specialist("scheduling", "Find the earliest follow-up slot.", c)

### The orchestrator: decompose → dispatch → aggregate → verify
The orchestrator owns the plan and, crucially, a **verification** step over the combined result.

In [3]:
def orchestrator(goal, context):
    print("ORCHESTRATOR goal:", goal)
    results = {"triage": triage_agent(context),
               "records": records_agent(context),
               "scheduling": scheduling_agent(context)}
    for k, v in results.items(): print(f"   [{k}] -> {v}")
    # aggregate
    plan = (f"Follow-up plan: {results['records']} "
            f"Urgency: {results['triage']} Next visit: {results['scheduling']} "
            "Action: draft dose-review flag for clinician; change no order.")
    # verify: does the evidence justify the urgency? (a cheap consistency check)
    ok = ("HIGH" in results["triage"]) == ("4.2" in results["records"])
    print("   [verify] evidence consistent with urgency:", ok)
    return plan

plan = orchestrator("Prepare the anticoagulant follow-up.", PATIENT)
print("\nAGGREGATED PLAN:\n ", plan)

ORCHESTRATOR goal: Prepare the anticoagulant follow-up.


   [triage] -> Urgency: High. INR of 4.2 indicates a significant risk of bleeding. Immediate medical evaluation is needed to adjust warfarin dosage and assess for potential complications.
   [records] -> INR: 4.2 (reference range 2.0-3.0)  
Medications:  
- Warfarin 5 mg daily  
- Lisinopril 10 mg daily
   [scheduling] -> The earliest follow-up slot should be within 1-2 days to reassess INR levels and adjust warfarin dosage. Please schedule for tomorrow or the day after.
   [verify] evidence consistent with urgency: False

AGGREGATED PLAN:
  Follow-up plan: INR: 4.2 (reference range 2.0-3.0)  
Medications:  
- Warfarin 5 mg daily  
- Lisinopril 10 mg daily Urgency: Urgency: High. INR of 4.2 indicates a significant risk of bleeding. Immediate medical evaluation is needed to adjust warfarin dosage and assess for potential complications. Next visit: The earliest follow-up slot should be within 1-2 days to reassess INR levels and adjust warfarin dosage. Please schedule for tomorrow or the 

### The same task with one agent
One model, all the context, a single call. Fewer moving parts, no inter-agent messaging.

In [4]:
single = chat([{"role":"system","content":"You are a clinical follow-up assistant. Produce the plan; change no order."},
               {"role":"user","content":f"Prepare the anticoagulant follow-up. Context: {PATIENT}"}])
print("SINGLE-AGENT PLAN:\n ", single)

SINGLE-AGENT PLAN:
  **Anticoagulant Follow-Up Plan:**

1. **Warfarin Adjustment:**
   - Hold the current dose of warfarin (5 mg daily) due to the elevated INR of 4.2.
   - Consider a dose reduction or temporary discontinuation based on clinical judgment and patient factors.

2. **Monitoring INR:**
   - Schedule a follow-up INR check within 1-2 days to monitor the INR levels closely after the dose adjustment.
   - Ensure that the patient understands the importance of regular INR monitoring.

3. **Patient Education:**
   - Educate the patient on signs and symptoms of bleeding (e.g., unusual bruising, blood in urine or stool, prolonged bleeding from cuts).
   - Discuss dietary considerations, particularly regarding vitamin K intake, and the importance of consistency in diet.

4. **Lisinopril Continuation:**
   - Continue lisinopril 10 mg daily as prescribed, monitoring for any potential interactions or side effects.

5. **Follow-Up Appointment:**
   - Schedule a follow-up appointment wit

### Takeaway
Both produce a usable plan. The multi-agent version buys **separation of concerns, parallelism, and a natural place to verify** — worth it when sub-tasks need different tools, context, or specialization. It also costs more calls, more latency, and more ways to fail (miscommunication between agents is a leading multi-agent failure mode). The honest default from the white paper: **start with one well-instrumented agent and add agents only when the work demands it.** Reach for orchestration for genuinely separable, specialized workloads — not by reflex.

*Try:* give one sub-agent its own tool the others lack (e.g., scheduling calls a calendar API), and the case for splitting gets stronger.